[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB01_1_Ejercicio_Guiado_Clasificacion_Multiclase.ipynb)

# NB01_1 — Ejercicio guiado: clasificación multiclase con una Red Neuronal

## Encuesta Nacional de Participación y Opinión Ciudadana 2026

En este notebook reutilizaremos el **mismo conjunto de datos ficticio utilizado en la Evaluación 1**, donde el desafío principal fue diagnosticar y preparar los datos.

Ahora cambia el propósito: **utilizaremos los datos para construir una Red Neuronal de clasificación multiclase**.

### Problema de Machine Learning

Queremos predecir la variable:

**`medio_informacion`**

que posee cuatro clases:

- Televisión
- Radio
- Prensa digital
- Redes sociales

Utilizaremos aproximadamente **cinco variables predictoras**.

> 🎯 **Objetivo del ejercicio:** comprender el flujo completo de una red neuronal multiclase.  
> El notebook guiará la programación, pero **las principales decisiones de hiperparámetros serán tomadas por el estudiante**.


## 🧭 Antes de comenzar: ¿qué decisiones tomarás tú?

Durante el ejercicio encontrarás una celda denominada **Zona de decisiones del estudiante**.

Allí deberás seleccionar, entre otros:

- número de neuronas de la capa oculta;
- función de activación;
- optimizador;
- `learning_rate`;
- número de épocas;
- `batch_size`.

Puedes apoyarte en el simulador desarrollado para el curso:

👉 [https://juliopez.github.io/Taller-Fundamentos-Data-Science-Python/Simulador_Redes_Neuronales.html](https://juliopez.github.io/Taller-Fundamentos-Data-Science-Python/Simulador_Redes_Neuronales.html)

> El simulador **no entrega una configuración “correcta”**. Su función es ayudarte a comprender las alternativas y justificar un punto de partida.


## 1. Importar las librerías

Usaremos:

- **Pandas / NumPy** para trabajar con los datos.
- **Scikit-learn** para separar, transformar y evaluar.
- **TensorFlow/Keras** para construir la red neuronal.
- **Matplotlib** para observar el entrenamiento.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense

print("TensorFlow:", tf.__version__)


## 2. Cargar el conjunto de datos

En la Evaluación 1 trabajamos con este dataset para detectar problemas de calidad y preparación.

Aquí volveremos a utilizarlo, pero el foco será distinto: **construir un modelo predictivo**.

La siguiente celda intenta cargar el archivo directamente desde GitHub. Si el nombre o ubicación del archivo cambia, puedes reemplazar `URL_DATASET` por la URL RAW correspondiente.


In [ ]:
URL_DATASET = "https://raw.githubusercontent.com/juliopez/Taller-Fundamentos-Data-Science-Python/refs/heads/main/Machine_Learning_2026/02_Datasets/02_Encuesta_Participacion_Opinion_Ciudadana_2026.csv"

df = pd.read_csv(URL_DATASET)

print("Dimensiones:", df.shape)
df.head()


## 3. Reconocer nuevamente los datos

Aunque ya conocemos el dataset, **un modelo nunca debería construirse sin verificar primero qué datos estamos utilizando**.

Observaremos:

- dimensiones;
- nombres de variables;
- tipos de datos;
- valores faltantes;
- distribución de la variable que queremos predecir.


In [ ]:
display(df.head())
print("\nColumnas:")
print(df.columns.tolist())

print("\nTipos:")
display(df.dtypes.to_frame("tipo"))

print("\nValores faltantes:")
display(df.isna().sum().to_frame("faltantes"))

print("\nDistribución de medio_informacion:")
display(df["medio_informacion"].value_counts(dropna=False).to_frame("frecuencia"))


## 4. Definir el problema

La variable dependiente será:

```text
y = medio_informacion
```

Usaremos cinco variables predictoras:

1. `edad`
2. `interes_politica`
3. `confianza_instituciones`
4. `ingreso_hogar`
5. `participacion_anterior`

### ¿Por qué no usamos `id_encuestado`?

Porque es un **identificador**, no una característica de la persona que deba utilizarse para aprender patrones.

### ¿Por qué no utilizamos todas las variables?

Porque este es un ejercicio guiado. Reducir el número de variables nos permite concentrarnos en la lógica de la Red Neuronal y en sus hiperparámetros.


In [ ]:
features = [
    "edad",
    "interes_politica",
    "confianza_instituciones",
    "ingreso_hogar",
    "participacion_anterior"
]

target = "medio_informacion"

X = df[features].copy()
y = df[target].copy()

print("X:", X.shape)
print("y:", y.shape)
display(X.head())


## 5. Preparación mínima de los datos

Una Red Neuronal necesita trabajar con valores numéricos.

Por eso realizaremos tres operaciones antes del entrenamiento:

- completar los valores faltantes de `edad`;
- transformar `participacion_anterior` a 0/1;
- transformar las cuatro clases de `medio_informacion` a etiquetas numéricas.

> 💡 Esta etapa conecta directamente con la Evaluación 1: **la calidad y preparación de los datos ocurre antes del modelamiento**.


In [ ]:
# Imputación de edad con la mediana
X["edad"] = X["edad"].fillna(X["edad"].median())

# Variable binaria
X["participacion_anterior"] = (
    X["participacion_anterior"]
    .astype(str)
    .str.strip()
    .map({"No": 0, "Sí": 1, "Si": 1})
)

# Eliminar eventualmente registros que no pudieron convertirse
mask = X["participacion_anterior"].notna() & y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].copy()

# Codificación de la variable objetivo
encoder_y = LabelEncoder()
y_encoded = encoder_y.fit_transform(y)

print("Clases y codificación:")
for i, clase in enumerate(encoder_y.classes_):
    print(f"{i} -> {clase}")

print("\nValores faltantes en X:", int(X.isna().sum().sum()))


## 6. Separar entrenamiento y prueba

Utilizaremos una partición **80 % entrenamiento / 20 % prueba**.

Además, aplicaremos `stratify=y_encoded` para conservar aproximadamente la proporción de las cuatro clases en ambos subconjuntos.

El conjunto de prueba quedará reservado para la evaluación final.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)


## 7. Estandarizar las variables predictoras

Las variables tienen escalas muy diferentes. Por ejemplo:

- `interes_politica`: 1–10;
- `confianza_instituciones`: 0–100;
- `ingreso_hogar`: cientos de miles o millones.

Utilizaremos `StandardScaler`.

### Regla importante

El escalador se **ajusta (`fit`) solamente con los datos de entrenamiento**.  
Después, esa misma transformación se aplica a entrenamiento y prueba.

Esto evita utilizar información del conjunto de prueba durante el aprendizaje.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Forma X_train_scaled:", X_train_scaled.shape)
print("Media aproximada de entrenamiento:")
print(np.round(X_train_scaled.mean(axis=0), 3))


# 🎛️ 8. Zona de decisiones del estudiante

Hasta aquí el notebook ha definido el **problema y la preparación de los datos**.

Ahora comienzan tus decisiones de arquitectura y entrenamiento.

Antes de modificar la siguiente celda, recuerda:

- La **cantidad de entradas** no se elige arbitrariamente: depende del número final de características que recibe el modelo.
- La **cantidad de neuronas de salida** tampoco se elige arbitrariamente en este ejercicio: tenemos 4 clases, por lo que utilizaremos 4 neuronas con `softmax`.
- La **cantidad de capas ocultas** sí es una decisión de diseño.
- La **cantidad de neuronas de cada capa oculta** también es una decisión de diseño.

Como punto de partida puedes comenzar con una arquitectura sencilla —por ejemplo, **1 capa oculta con 32 neuronas**— y luego experimentar con una segunda configuración.

### Tus decisiones

Deberás seleccionar:

- cuántas capas ocultas utilizar;
- cuántas neuronas tendrá cada capa oculta;
- función de activación de las capas ocultas;
- optimizador;
- `learning_rate`;
- número de épocas;
- `batch_size`.

👉 Puedes consultar el **simulador del curso** para fundamentar estas decisiones.

> ✍️ **No existe una única combinación correcta.** Lo importante es seleccionar una configuración razonable, ejecutarla, observar entrenamiento y validación, y justificar posteriormente las decisiones tomadas.


In [8]:
# ============================================================
# ZONA DE DECISIONES DEL ESTUDIANTE
# Modifica estos hiperparámetros antes de entrenar.
# ============================================================

# 1) ¿Cuántas capas ocultas utilizarás?
# En este ejercicio puedes trabajar con 1, 2 o 3.
CAPAS_OCULTAS =   ????? #1

# 2) ¿Cuántas neuronas tendrá cada capa?
# Solo se utilizarán las capas indicadas en CAPAS_OCULTAS.
NEURONAS_CAPA_1 = ????? #32
NEURONAS_CAPA_2 = ????? #16
NEURONAS_CAPA_3 = ????? #8

# 3) Activación de las capas ocultas
ACTIVACION =      ????? #"relu"

# 4) Configuración del entrenamiento
OPTIMIZADOR =     ????? #"adam"
LEARNING_RATE =   ????? #0.001
EPOCHS =          ????? #50
BATCH_SIZE =      ????? #32


## 9. Construir la arquitectura

Ahora traduciremos las decisiones anteriores a una arquitectura Keras.

La estructura general será:

**Entrada → capa(s) oculta(s) → salida**

### Entrada

La entrada queda determinada automáticamente por:

```python
n_features = X_train_scaled.shape[1]
```

Por tanto, no necesitamos escribir manualmente cuántas entradas existen.

> 💡 Recuerda: el número de variables conceptuales y el número final de características no siempre coinciden. Una transformación como *one-hot encoding* puede convertir una variable categórica en varias columnas.

### Capas ocultas

Aquí sí intervienen tus decisiones:

- `CAPAS_OCULTAS`
- `NEURONAS_CAPA_1`
- `NEURONAS_CAPA_2`
- `NEURONAS_CAPA_3`
- `ACTIVACION`

El código agregará automáticamente solamente las capas que hayas seleccionado.

### Salida

Nuestro problema tiene **4 clases excluyentes** de `medio_informacion`.

Por eso la salida tendrá:

**4 neuronas → `softmax`**

La cantidad de neuronas de salida se obtendrá automáticamente mediante `n_clases`.


In [ ]:
n_features = X_train_scaled.shape[1]
n_clases = len(encoder_y.classes_)

if CAPAS_OCULTAS not in [1, 2, 3]:
    raise ValueError("CAPAS_OCULTAS debe ser 1, 2 o 3.")

modelo = Sequential()
modelo.add(Input(shape=(n_features,)))

# Primera capa oculta: siempre existe
modelo.add(Dense(NEURONAS_CAPA_1, activation=ACTIVACION))

# Capas adicionales: se agregan solo si fueron seleccionadas
if CAPAS_OCULTAS >= 2:
    modelo.add(Dense(NEURONAS_CAPA_2, activation=ACTIVACION))

if CAPAS_OCULTAS >= 3:
    modelo.add(Dense(NEURONAS_CAPA_3, activation=ACTIVACION))

# Capa de salida: determinada por las 4 clases del problema
modelo.add(Dense(n_clases, activation="softmax"))

print("Características de entrada:", n_features)
print("Capas ocultas seleccionadas:", CAPAS_OCULTAS)
print("Clases de salida:", n_clases)
print()

modelo.summary()


## 10. Configurar el entrenamiento

`compile()` no agrega nuevas capas.

Aquí definimos **cómo aprenderá el modelo**:

- el optimizador;
- el `learning_rate`;
- la función de pérdida;
- la métrica que observaremos.

Como nuestras clases fueron codificadas como enteros (`0`, `1`, `2`, `3`), utilizaremos:

```python
sparse_categorical_crossentropy
```


In [ ]:
optimizadores = {
    "adam": tf.keras.optimizers.Adam,
    "sgd": tf.keras.optimizers.SGD,
    "rmsprop": tf.keras.optimizers.RMSprop,
    "adamw": tf.keras.optimizers.AdamW,
    "adagrad": tf.keras.optimizers.Adagrad,
    "adadelta": tf.keras.optimizers.Adadelta,
    "adamax": tf.keras.optimizers.Adamax,
    "nadam": tf.keras.optimizers.Nadam,
    "ftrl": tf.keras.optimizers.Ftrl
}

nombre_opt = OPTIMIZADOR.lower()

if nombre_opt not in optimizadores:
    raise ValueError(
        f"Optimizador '{OPTIMIZADOR}' no contemplado en este notebook. "
        f"Opciones: {list(optimizadores.keys())}"
    )

optimizer = optimizadores[nombre_opt](learning_rate=LEARNING_RATE)

modelo.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Optimizador:", OPTIMIZADOR)
print("Learning rate:", LEARNING_RATE)


## 11. Entrenar la Red Neuronal

Durante el entrenamiento reservaremos un **20 % del conjunto de entrenamiento como validación**.

Esto permite comparar:

- `accuracy` de entrenamiento;
- `val_accuracy` de validación;
- `loss` de entrenamiento;
- `val_loss` de validación.

Estas curvas serán fundamentales para interpretar qué ocurrió con los hiperparámetros seleccionados.


In [ ]:
tf.keras.utils.set_random_seed(42)

historial = modelo.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)


## 12. Observar el aprendizaje

No basta con mirar solamente el último `accuracy`.

Las curvas permiten observar si:

- el modelo sigue aprendiendo;
- entrenamiento y validación evolucionan de manera similar;
- aparece una separación importante entre ambas curvas;
- la pérdida de validación comienza a empeorar.

> 🔎 La pregunta relevante no es solamente **“¿qué accuracy obtuve?”**, sino también **“¿cómo aprendió el modelo?”**


In [ ]:
history = pd.DataFrame(historial.history)

plt.figure(figsize=(8, 4))
plt.plot(history["accuracy"], label="Entrenamiento")
plt.plot(history["val_accuracy"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.title("Accuracy durante el entrenamiento")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history["loss"], label="Entrenamiento")
plt.plot(history["val_loss"], label="Validación")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Loss durante el entrenamiento")
plt.legend()
plt.show()


## 13. Evaluar con datos no utilizados en el entrenamiento

Ahora utilizaremos `X_test`.

Este conjunto no participó en el ajuste del escalador ni en el entrenamiento de la red.

Por eso nos permite obtener una estimación más honesta del comportamiento del modelo sobre observaciones no utilizadas para aprender.


In [ ]:
test_loss, test_accuracy = modelo.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print(f"Loss en prueba:     {test_loss:.4f}")
print(f"Accuracy en prueba: {test_accuracy:.4f}")


## 14. Convertir probabilidades en clases

Softmax entrega una probabilidad para cada una de las cuatro clases.

`argmax()` selecciona la posición con la probabilidad más alta.

Después utilizaremos `inverse_transform()` para recuperar los nombres originales de las categorías.


In [ ]:
probabilidades = modelo.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(probabilidades, axis=1)

resultado = pd.DataFrame({
    "Real": encoder_y.inverse_transform(y_test),
    "Predicho": encoder_y.inverse_transform(y_pred),
    "Probabilidad_max": probabilidades.max(axis=1)
})

resultado.head(10)


## 15. Matriz de confusión

La matriz de confusión permite observar **qué clases se confunden entre sí**.

Esto entrega más información que una única cifra de `accuracy`.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=encoder_y.classes_
)

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, values_format="d")
plt.xticks(rotation=30, ha="right")
plt.title("Matriz de confusión")
plt.show()


## 16. Comparar con una referencia simple

Antes de concluir que una Red Neuronal “funciona bien”, necesitamos contexto.

Calcularemos qué accuracy obtendríamos si siempre predijéramos la clase más frecuente del conjunto de entrenamiento.

Esta referencia se denomina aquí **baseline simple**.

> Si una red compleja apenas supera —o no supera— una estrategia extremadamente simple, eso también es un resultado que debemos interpretar.


In [ ]:
clase_mayoritaria = np.bincount(y_train).argmax()
baseline_pred = np.full_like(y_test, clase_mayoritaria)

baseline_accuracy = accuracy_score(y_test, baseline_pred)

print("Clase mayoritaria:",
      encoder_y.inverse_transform([clase_mayoritaria])[0])
print(f"Accuracy baseline: {baseline_accuracy:.4f}")
print(f"Accuracy red:      {test_accuracy:.4f}")


# ✍️ 17. Reflexión del estudiante

Completa esta celda **después de ejecutar el modelo**.

### Arquitectura utilizada

- Número de características de entrada:
- Cantidad de capas ocultas:
- Neuronas en la capa oculta 1:
- Neuronas en la capa oculta 2 (si corresponde):
- Neuronas en la capa oculta 3 (si corresponde):
- Número de neuronas de salida:
- Activación de las capas ocultas:

### Configuración del entrenamiento

- Optimizador:
- Learning rate:
- Epochs:
- Batch size:

### Justificación

Explica brevemente:

1. ¿Por qué seleccionaste esa cantidad de capas ocultas?
2. ¿Por qué seleccionaste esa cantidad de neuronas?
3. ¿Por qué seleccionaste esa función de activación?
4. ¿Qué criterio utilizaste para los hiperparámetros de entrenamiento?

### Resultados

- Accuracy de entrenamiento final:
- Accuracy de validación final:
- Accuracy de prueba:
- Accuracy baseline:

### Interpretación

1. ¿La red supera el baseline?
2. ¿Observas señales de overfitting o underfitting?
3. ¿Qué modificarías primero: cantidad de capas, neuronas u otro hiperparámetro?
4. ¿Qué esperas que ocurra al modificarlo?


# 🔁 18. Segundo experimento

Ahora modifica **una o dos decisiones de arquitectura o entrenamiento** en la sección 8 y vuelve a ejecutar desde la construcción del modelo.

Por ejemplo, puedes:

- aumentar o disminuir las neuronas;
- agregar una segunda capa oculta;
- cambiar la activación;
- modificar `learning_rate`, `epochs` o `batch_size`.

Registra los resultados:

| Experimento | Capas ocultas | Neuronas | Activación | Optimizador | Learning rate | Epochs | Batch | Accuracy prueba |
|---|---:|---|---|---|---:|---:|---:|---:|
| 1 |  |  |  |  |  |  |  |  |
| 2 |  |  |  |  |  |  |  |  |

### Pregunta final

**¿Cuál de las dos configuraciones conservarías y qué evidencia utilizarías para justificar tu decisión?**

> El propósito no es “adivinar” la arquitectura correcta. El propósito es formular una configuración, observar evidencia, comparar y tomar una decisión fundamentada.


## ✅ Cierre

En este ejercicio recorrimos el flujo:

**datos → preparación → variables X/y → train/test → escalamiento → arquitectura → hiperparámetros → entrenamiento → validación → prueba → interpretación**

En la próxima evaluación, este flujo será la referencia conceptual, pero deberás tomar decisiones con mayor autonomía.
